Models:
- Logistic Regression
- Tree Decision
- SVC
- Random Forest
- CatBoost
- XGBoost

For imbalance fix:
- `class-weight`
- `undersampling` & `oversampling`
- `SMOTE`

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from imblearn.over_sampling import SMOTENC
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (precision_score,
                             recall_score,
                             average_precision_score,
                             f1_score)

import joblib

In [2]:
RANDOM_SEED = 42
df = pd.read_csv('../data/Synthetic_Financial_datasets_log.csv')

## New Features

In [3]:
df['log_amount'] = np.log1p(df['amount'])
df['day'] = (df['step'] - 1) // 24
df['hour'] = (df['step'] - 1) % 24
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

features = ['step', 'type',
            'amount', 'log_amount',
            'day', 'hour_sin',
            'hour_cos']

target = ['isFraud']

df = df[features + target]

In [4]:
test_point = 550
calibration_point = test_point - 50
validation_point = calibration_point - 100

test = df[df['step'] > test_point].copy()
calibration = df[(df['step'] <= test_point) & (df['step'] > calibration_point)].copy()
validation = df[(df['step'] <= calibration_point) & (df['step'] > validation_point)].copy()
train = df[df['step'] <= validation_point].copy()

In [5]:
X_train, y_train = train.drop('isFraud', axis=1), train['isFraud']
X_val, y_val = validation.drop('isFraud', axis=1), validation['isFraud']
X_cal, y_cal = calibration.drop('isFraud', axis=1), calibration['isFraud']
X_test, y_test = test.drop('isFraud', axis=1), test['isFraud']

# For SMOTENC
X_train['type'] = X_train['type'].astype('category')

In [6]:
categorial_features = ['type']
numeric_features = ['amount', 'log_amount']
other_features =['step', 'day', 'hour_sin', 'hour_cos']

## Pipelines

### Logistic Regression

#### Preprocessing

In [7]:
log_reg_preprocessor = ColumnTransformer(transformers=[
    ('numeric', StandardScaler(), numeric_features),
    ('categorical', OneHotEncoder(), categorial_features),
    ('others', 'passthrough', other_features)
])

#### Pipelines

In [8]:
log_reg_pipelines = {
    'log_reg_pipeline_base': Pipeline(steps=[
        ('preprocessor', log_reg_preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_SEED)
        )
    ]),

    'log_reg_pipeline_balanced': Pipeline(steps=[
        ('preprocessor', log_reg_preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_SEED)
        )
    ]),

    # UnderSampling
    'log_reg_pipeline_under.05': ImbPipeline(steps=[
        ('preprocessor', log_reg_preprocessor),
        ('undersampling', RandomUnderSampler(
            sampling_strategy=0.05,
            random_state=RANDOM_SEED)),
        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_SEED)
        )
    ]),

    'log_reg_pipeline_under.10': ImbPipeline(steps=[
        ('preprocessor', log_reg_preprocessor),
        ('undersampling', RandomUnderSampler(
            sampling_strategy=0.10,
            random_state=RANDOM_SEED)),
        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_SEED)
        )
    ]),

    'log_reg_pipeline_under.20': ImbPipeline(steps=[
        ('preprocessor', log_reg_preprocessor),
        ('undersampling', RandomUnderSampler(
            sampling_strategy=0.20,
            random_state=RANDOM_SEED)),
        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_SEED)
        )
    ]),

    # OverSampling
    'log_reg_pipeline_over.005': ImbPipeline(steps=[
        ('preprocessor', log_reg_preprocessor),
        ('undersampling', RandomOverSampler(
            sampling_strategy=0.005,
            random_state=RANDOM_SEED)),
        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_SEED)
        )
    ]),

    'log_reg_pipeline_over.01': ImbPipeline(steps=[
        ('preprocessor', log_reg_preprocessor),
        ('undersampling', RandomOverSampler(
            sampling_strategy=0.01,
            random_state=RANDOM_SEED)),
        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_SEED)
        )
    ]),

    'log_reg_pipeline_over.02': ImbPipeline(steps=[
        ('preprocessor', log_reg_preprocessor),
        ('undersampling', RandomOverSampler(
            sampling_strategy=0.02,
            random_state=RANDOM_SEED)),
        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_SEED)
        )
    ]),

    'log_reg_pipeline_smote.01': ImbPipeline(steps=[
        ('smote', SMOTENC(
            categorical_features=categorial_features,
            sampling_strategy=0.01,
            random_state=RANDOM_SEED,
            k_neighbors=5
        )),
        ('preprocessor', log_reg_preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_SEED)
        )
    ]),

    'log_reg_pipeline_smote.02': ImbPipeline(steps=[
        ('smote', SMOTENC(
            categorical_features=categorial_features,
            sampling_strategy=0.02,
            random_state=RANDOM_SEED,
            k_neighbors=5
        )),
        ('preprocessor', log_reg_preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_SEED)
        )
    ]),
}

In [9]:
res_val_table = pd.DataFrame(
    columns=[
        'model',
        'precision',
        'recall',
        'f1',
        'average_precision'
    ]
)

In [10]:
def estimate_model(y_true, y_proba):
    y_pred = (y_proba >= 0.5).astype(int)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    average_precision = average_precision_score(y_true, y_proba)

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'average_precision': average_precision,
    }

In [11]:
for name, model in log_reg_pipelines.items():
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_val)[:, 1]
    metrics = estimate_model(y_val, y_proba)
    metrics['model'] = name
    res_val_table = pd.concat([res_val_table, pd.DataFrame([metrics])], ignore_index=True)

In [12]:
res_val_table

,model,precision,recall,f1,average_precision
0,log_reg_pipeline_base,1.0,0.009225,0.018282,0.239316
1,log_reg_pipeline_balanced,0.028503,0.830258,0.055113,0.216937
2,log_reg_pipeline_under.05,0.028635,0.835793,0.055372,0.212705
3,log_reg_pipeline_under.10,0.028408,0.835793,0.054949,0.216078
4,log_reg_pipeline_under.20,0.02856,0.835793,0.055232,0.21451
5,log_reg_pipeline_over.005,0.027725,0.835793,0.05367,0.215862
6,log_reg_pipeline_over.01,0.027873,0.835793,0.053948,0.215848
7,log_reg_pipeline_over.02,0.028218,0.833026,0.054587,0.216732
8,log_reg_pipeline_smote.01,0.027174,0.667897,0.052223,0.085043
9,log_reg_pipeline_smote.02,0.027111,0.650369,0.052053,0.081631
